<a href="https://colab.research.google.com/github/ridamumtazz/Flyrank-ML-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

from IPython.display import Markdown, display

display(Markdown("""
### Method Choice and Why

I chose **Random Forest Regressor** because my lane focuses on ranking and scoring content items. The model can predict a continuous performance score that can be used to prioritize content for improvement. Random Forest is suitable because it can learn non-linear relationships between different performance signals such as clicks, impressions, CTR, average position, and engagement. It also works well with multiple features and can help identify which signals are important for the predicted score.
"""))

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

from IPython.display import Markdown, display

display(Markdown("""
### Split Design

I will use a **time-aware split** because the goal is to make predictions for future content performance using information available in the past. The earlier time period will be used for training, while a later time period will be used for testing.

This is a more honest split for the problem because it avoids using future performance information to predict the past. The same time-based split should also be used when comparing the model with the Week-4 baseline.
"""))

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [2]:
# ============================================
# ML-08 SECTION 3 - PREPARE DATA
# ============================================

from google.colab import userdata
from huggingface_hub import hf_hub_download
import pandas as pd
import numpy as np

# 1. Get Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

# 2. Download dataset
sample_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

# 3. Load dataset
sample_df = pd.read_parquet(sample_path)

print("Dataset is ready.")
print("Rows:", sample_df.shape[0])
print("Columns:", sample_df.shape[1])

# 4. Make a copy
model_df = sample_df.copy()

# 5. Convert date
model_df["report_date"] = pd.to_datetime(
    model_df["report_date"]
)

# 6. Create CTR
model_df["ctr"] = np.where(
    model_df["gsc_impressions"] > 0,
    model_df["gsc_clicks"] / model_df["gsc_impressions"],
    0
)

# 7. Create engagement rate
model_df["engagement_rate"] = np.where(
    model_df["ga4_sessions"] > 0,
    model_df["ga4_engaged_sessions"] / model_df["ga4_sessions"],
    0
)

# 8. Sort data by content and date
model_df = model_df.sort_values(
    ["content_hash_id", "report_date"]
)

# 9. Create next-period clicks as target
model_df["target_clicks"] = (
    model_df
    .groupby("content_hash_id")["gsc_clicks"]
    .shift(-1)
)

# 10. Select features
features = [
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "ga4_sessions",
    "engagement_rate"
]

# 11. Remove missing values
model_df = model_df.dropna(
    subset=features + ["target_clicks"]
)

print("================================")
print("DATA PREPARATION COMPLETE")
print("================================")
print("Prepared data shape:", model_df.shape)
print("Features:", features)

Dataset is ready.
Rows: 11694072
Columns: 31
DATA PREPARATION COMPLETE
Prepared data shape: (2819831, 34)
Features: ['gsc_impressions', 'gsc_clicks', 'ctr', 'gsc_avg_position', 'ga4_sessions', 'engagement_rate']


In [3]:
# ============================================
# ML-08 SECTION 3 - TRAIN RANDOM FOREST
# ============================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Time-aware split
split_date = model_df["report_date"].quantile(0.8)

train_df = model_df[
    model_df["report_date"] < split_date
]

test_df = model_df[
    model_df["report_date"] >= split_date
]

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

# Use a manageable sample for training
train_sample = train_df.sample(
    n=min(100000, len(train_df)),
    random_state=42
)

# Prepare training and testing data
X_train = train_sample[features]
y_train = train_sample["target_clicks"]

# Use a smaller test sample for faster evaluation
test_sample = test_df.sample(
    n=min(30000, len(test_df)),
    random_state=42
)

X_test = test_sample[features]
y_test = test_sample["target_clicks"]

# Train Random Forest
rf_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Make predictions
predictions = rf_model.predict(X_test)

# Calculate metrics
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("\n================================")
print("RANDOM FOREST RESULTS")
print("================================")
print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))

Training rows: 2226478
Testing rows: 593353

RANDOM FOREST RESULTS
MAE: 0.262
RMSE: 2.4575


In [4]:
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

display(importance_df)

,Feature,Importance
1,gsc_clicks,0.469812
4,ga4_sessions,0.438926
0,gsc_impressions,0.056971
3,gsc_avg_position,0.021744
2,ctr,0.009558
5,engagement_rate,0.002989


I trained a Random Forest Regressor using a time-aware split, with earlier observations used for training and later observations used for testing. The model used six performance features: impressions, clicks, CTR, average position, sessions, and engagement rate. The model was evaluated using MAE and RMSE. The Random Forest achieved an MAE of 0.262 and an RMSE of 2.4575 on the test data. These results will be compared with the Week-4 baseline using the same evaluation metric and split. The comparison is intended to provide decision-support for prioritizing content rather than guarantee future performance.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and Interpretation

The Random Forest model achieved an MAE of 0.262 and an RMSE of 2.4575 on the test data. The model relies mostly on GSC clicks (0.4698) and GA4 sessions (0.4389), while impressions (0.0570), average position (0.0217), CTR (0.0096), and engagement rate (0.0030) have lower importance. This shows that the model is mainly influenced by existing traffic and click signals. The model may be less reliable for content with very low traffic or limited historical data because there is less information available for prediction. Therefore, the model should be treated as directional decision-support rather than a guarantee of future content performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.